In [ ]:
import pandas as pd

df = pd.read_parquet("../data/processed/wti_panel_2021_2025.parquet")

df.head(12)


In [ ]:
from commodity_factors.simulation import simulate_states, simulate_dataset 

import numpy as np
params = {
    "kappa": 1.5,
    "sigma_chi": 0.30,
    "sigma_xi": 0.20,
    "rho": 0.30,
    "mu_xi": 0.02,
    "lambda_chi": 0.01,
    "lambda_xi": 0.01,
}

tau = np.array([1, 2, 3, 4, 5, 6]) / 12

states, log_prices = simulate_dataset(
    n_steps=1000,
    tau=tau,
    params=params,
)

print("States:", states.shape)
print("Log futures prices:", log_prices.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

for j in range(log_prices.shape[1]):
    plt.plot(np.exp(log_prices[:, j]), label=f"Maturity {j+1}")

plt.xlabel("Time step")
plt.ylabel("Futures price ($)")
plt.legend()
plt.show()

In [ ]:
from commodity_factors.kalman import kalman_filter

results = kalman_filter(
    log_prices=log_prices,
    tau=tau,
    params=params,
)

estimated_states = results["filtered_states"]

print("True states:", states.shape)
print("Estimated states:", estimated_states.shape)
print("Log-likelihood:", results["log_likelihood"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(states[:, 0], label="True chi")
axes[0].plot(estimated_states[:, 0], label="Estimated chi")
axes[0].set_ylabel("Short-term factor")
axes[0].legend()

axes[1].plot(states[:, 1], label="True xi")
axes[1].plot(estimated_states[:, 1], label="Estimated xi")
axes[1].set_ylabel("Long-term factor")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
rmse = np.sqrt(
    np.mean((states - estimated_states) ** 2, axis=0)
)

print(f"Chi RMSE: {rmse[0]:.6f}")
print(f"Xi RMSE:  {rmse[1]:.6f}")

In [ ]:
from commodity_factors.estimation import estimate_parameters

estimated = estimate_parameters(
    log_prices=log_prices,
    tau=tau,
    fixed_params=params,
)

print("Optimization successful:", estimated["success"])
print("Message:", estimated["message"])

for name in ["kappa", "sigma_chi", "sigma_xi", "rho"]:
    print(
        f"{name:12s} "
        f"True: {params[name]:.4f}  "
        f"Estimated: {estimated['params'][name]:.4f}"
    )

In [ ]:
estimated_params = estimated["params"]

results_mle = kalman_filter(
    log_prices=log_prices,
    tau=tau,
    params=estimated_params,
)

estimated_states_mle = results_mle["filtered_states"]

In [ ]:
import pandas as pd

parameter_names = ["kappa", "sigma_chi", "sigma_xi", "rho"]

comparison = pd.DataFrame({
    "True": [params[p] for p in parameter_names],
    "Estimated": [estimated_params[p] for p in parameter_names],
})

comparison["Error"] = comparison["Estimated"] - comparison["True"]

print("Optimization successful:", estimated["success"])
display(comparison)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

factor_names = ["Short-term factor (chi)", "Long-term factor (xi)"]

for i, ax in enumerate(axes):
    ax.plot(states[:, i], label="True", linewidth=2)
    ax.plot(
        estimated_states[:, i],
        label="Kalman (true parameters)",
        alpha=0.7,
    )
    ax.plot(
        estimated_states_mle[:, i],
        label="Kalman (MLE parameters)",
        alpha=0.7,
        linestyle="--",
    )

    ax.set_ylabel(factor_names[i])
    ax.legend()

axes[-1].set_xlabel("Simulation day")
plt.tight_layout()
plt.show()

In [ ]:
rmse_true_params = np.sqrt(
    np.mean((states - estimated_states) ** 2, axis=0)
)

rmse_mle_params = np.sqrt(
    np.mean((states - estimated_states_mle) ** 2, axis=0)
)

rmse_comparison = pd.DataFrame(
    {
        "True parameters": rmse_true_params,
        "MLE parameters": rmse_mle_params,
    },
    index=["Chi", "Xi"],
)

display(rmse_comparison)

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/wti_panel_2021_2025.parquet"
)

print(df.columns.tolist())
print(df.head())

In [ ]:
import numpy as np
from commodity_factors.kalman import kalman_filter

results_scalar = kalman_filter(
    log_prices, tau, params, dt=1 / 252
)

results_array = kalman_filter(
    log_prices, tau, params,
    dt=np.full(len(log_prices) - 1, 1 / 252)
)

np.testing.assert_allclose(
    results_scalar["filtered_states"],
    results_array["filtered_states"],
)

print("Test passed!")